## WinREd

In [ ]:
.open tmp.db

drop table if exists temp.trump24_donations;
create table temp.trump24_donations as 
  select 
    filing_id,
    transaction_id,
    format(
      '%s-%s-%s-%s',
      lower(contributor_first_name),
      lower(contributor_last_name),
      substr(contributor_zip_code, 1, 5),
      contributor_state
    ) as contributor_id,
    contribution_date,
    -- 'Earmarked for REPUBLICAN NATIONAL COMMITTEE (C00003418)' -> 'C00003418'
    regex_find(
      'C\d+', 
      contribution_purpose_descrip
    ) as earmark_committee_id,
    substr(contributor_zip_code, 1, 5) as contributor_zip5,
    contributor_state,
    contributor_employer,
    contributor_occupation,
    contribution_amount
  from libfec_schedule_a
  where earmark_committee_id in (
    select value
    from json_each('[
      "C00867937", // Trump 47
      "C00873893", // Trump National Committee
      "C00828541", // Donald J. Trump for President 2024 (renamed?
      "C00762591", // Save America
      "C00580100", // Make America Great Again PAC
      "C00770941", // Trump Save America JFC
      "C00618389", // Trump Victory
      "C00618371", // Trump Make America Great Again Committee
      "C00855114", // Trump Bilirakis Victory Fund
      "C00003418", // Republican National Committee
    ]')
  );

select count(*) from temp.trump24_donations;

count(*)
17284275


In [ ]:
select count(distinct contributor_id) 
from temp.trump24_donations
where contribution_date = '2024-05-31';

count(distinct contributor_id)
373198


In [25]:

select
  count(distinct contributor_id),
  sum(contribution_amount)
from temp.trump24_donations
where contributor_zip5 = '85044'
  and contribution_date >= '2022-11-15';

count(distinct contributor_id),sum(contribution_amount)
532,102864.17


In [24]:
select min(contribution_date), max(contribution_date) from temp.trump24_donations;

min(contribution_date),max(contribution_date)
2022-10-20,2024-09-30


In [ ]:
select
  contribution_date,
  count(distinct contributor_id) 
from temp.trump24_donations
where contribution_date >= '2024-01-01'
group by 1
order by 1;

## ActBlue

In [1]:
.open actblue.db

[no code]

In [4]:
select memo_text_description from libfec_schedule_a
where memo_text_description like '%Biden Victory Fund%'
limit 10;

memo_text_description
Earmarked for BIDEN VICTORY FUND (C00744946)
Earmarked for BIDEN VICTORY FUND (C00744946)
Earmarked for BIDEN VICTORY FUND (C00744946)
Earmarked for BIDEN VICTORY FUND (C00744946)
Earmarked for BIDEN VICTORY FUND (C00744946)
Earmarked for BIDEN VICTORY FUND (C00744946)
Earmarked for BIDEN VICTORY FUND (C00744946)
Earmarked for BIDEN VICTORY FUND (C00744946)
Earmarked for BIDEN VICTORY FUND (C00744946)
Earmarked for BIDEN VICTORY FUND (C00744946)


In [6]:
drop table if exists temp.harris24_donations;
create table temp.harris24_donations as 
  select 
    filing_id,
    transaction_id,
    format(
      '%s-%s-%s-%s',
      lower(contributor_first_name),
      lower(contributor_last_name),
      substr(contributor_zip_code, 1, 5),
      contributor_state
    ) as contributor_id,
    contribution_date,
    -- 'Earmarked for REPUBLICAN NATIONAL COMMITTEE (C00003418)' -> 'C00003418'
    regex_find(
      'C\d+', 
      memo_text_description
    ) as earmark_committee_id,
    substr(contributor_zip_code, 1, 5) as contributor_zip5,
    contributor_state,
    contributor_employer,
    contributor_occupation,
    contribution_amount
  from libfec_schedule_a
  where earmark_committee_id in (
    select value
    from json_each('[
      "C00703975", // Harris for President / Biden for President
      "C00838912", // Harris Action Fund
      "C00744946", // Harris Victory Fund / Biden Victory Fund
      "C00658476", // Democratic Grassroots Victory Fund
      "C00010603", // Democratic National Committee
    ]')
  );

┌├

In [13]:
select * from libfec_schedule_a where memo_text_description like '%Harris Action Fund%' limit 1;

filing_id,form_type,filer_committee_id_number,transaction_id,back_reference_tran_id_number,back_reference_sched_name,entity_type,contributor_organization_name,contributor_last_name,contributor_first_name,contributor_middle_name,contributor_prefix,contributor_suffix,contributor_street_1,contributor_street_2,contributor_city,contributor_state,contributor_zip_code,election_code,election_other_description,contribution_date,contribution_amount,contribution_aggregate,contribution_purpose_descrip,contributor_employer,contributor_occupation,donor_committee_fec_id,donor_committee_name,donor_candidate_fec_id,donor_candidate_last_name,donor_candidate_first_name,donor_candidate_middle_name,donor_candidate_prefix,donor_candidate_suffix,donor_candidate_office,donor_candidate_state,donor_candidate_district,conduit_name,conduit_street1,conduit_street2,conduit_city,conduit_state,conduit_zip_code,memo_code,memo_text_description,reference_code


In [7]:
select count(*) from temp.harris24_donations;

count(*)
18025158


In [10]:
select earmark_committee_id, count(*) from temp.harris24_donations
group by 1;

earmark_committee_id,count(*)
C00010603,2801289
C00658476,27
C00703975,7058573
C00744946,8165269


In [12]:
select * from temp.harris24_donations where earmark_committee_id = 'C00658476' limit 10;

filing_id,transaction_id,contributor_id,contribution_date,earmark_committee_id,contributor_zip5,contributor_state,contributor_employer,contributor_occupation,contribution_amount
1671120,SA11AI_500900303,charles-manning-12054-NY,2022-11-02,C00658476,12054,NY,NOT EMPLOYED,NOT EMPLOYED,50
1671120,SA11AI_499679528,geoff-trafton-34683-FL,2022-10-31,C00658476,34683,FL,NEW YORK LIFE,COMPUTER PROGRAMMER,25
1671120,SA11AI_499086913,robin-west-32606-FL,2022-10-29,C00658476,32606,FL,NOT EMPLOYED,NOT EMPLOYED,250
1686601,SA11AI_509442394,geoff-trafton-34683-FL,2022-11-30,C00658476,34683,FL,NEW YORK LIFE,COMPUTER PROGRAMMER,25
1686601,SA11AI_513130108,geoff-trafton-34683-FL,2022-12-31,C00658476,34683,FL,NEW YORK LIFE,COMPUTER PROGRAMMER,25
1720554,SA11AI_516562664,jay-liang-92336-CA,2023-01-30,C00658476,92336,CA,NOT EMPLOYED,NOT EMPLOYED,36500
1720554,SA11AI_533927721,albert-marrero-93001-CA,2023-06-23,C00658476,93001,CA,NOT EMPLOYED,NOT EMPLOYED,25
1720554,SA11AI_516604593,geoff-trafton-34683-FL,2023-01-31,C00658476,34683,FL,NEW YORK LIFE,COMPUTER PROGRAMMER,25
1720554,SA11AI_519685362,geoff-trafton-34683-FL,2023-02-28,C00658476,34683,FL,NEW YORK LIFE,COMPUTER PROGRAMMER,25
1720554,SA11AI_523413566,geoff-trafton-34683-FL,2023-03-31,C00658476,34683,FL,NEW YORK LIFE,COMPUTER PROGRAMMER,25


In [16]:
-- should be 651197
select count(distinct contributor_id)
from temp.harris24_donations
where contribution_date = '2024-07-22';

count(distinct contributor_id)
636665


In [5]:
select distinct earmark_committee_id from temp.harris24_donations;

SQL logic error: error[1]: no such table: temp.harris24_donations

